In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus(n_gpus=1)

import scanpy as sc
from essential.steady_state import SteadyStateEstimator
import numpy as np
import pandas as pd
import plotnine as gg
import matplotlib.pyplot as plt
import os


import scipy.stats as st


def test_model(estimator, adata, adata_test, n_steps=100):
    X_ctrl_avg = adata[adata.obs["consensus_target"] == "nontargeting"].X.mean(0).A1
    adata_ctrl = adata[adata.obs["consensus_target"] == "nontargeting"].copy()

    corrs = []
    errors = []
    heldout_perturbations = adata_test.obs["consensus_target"].unique()
    for g in heldout_perturbations:
        gene_idx = adata.var_names.get_loc(g)
        u_ = np.zeros(adata.n_vars)
        u_[gene_idx] = 1

        adata_test_u = adata_test[adata_test.obs["consensus_target"] == g].copy()
        X_gt_avg = adata_test_u.X.mean(0).A1
        X_pred = estimator.predict(adata_ctrl, u_, n_steps=n_steps)
        X_pred_avg = X_pred.mean(0)

        # normalize by control expression for Pearson correlations
        delta_X_pred = X_pred_avg - X_ctrl_avg
        delta_X_gt = X_gt_avg - X_ctrl_avg
        corrs.append(st.pearsonr(delta_X_pred, delta_X_gt)[0])

        # L2 errors without normalization
        errors.append(np.linalg.norm(X_pred_avg - X_gt_avg))
    return pd.DataFrame({"correlation": corrs, "error": errors})

In [ ]:
from essential.utils import load_regulondb_full


def build_aweight(
    adata: sc.AnnData, heldout_targets: list[str], perc_targets_in_training: float, random_seed: int
):
    ref_db = load_regulondb_full()
    ref_db_ = ref_db.set_index("tf_promoter")

    targets_in_literature_ = ref_db_["target_gene"].unique()
    targets_in_literature_ = [t.lower() for t in targets_in_literature_]
    var_names_ = [v.lower() for v in adata.var_names]
    covered_targets_ = np.intersect1d(targets_in_literature_, var_names_)

    heldout_targets_ = [t.lower() for t in heldout_targets]
    valid_targets = np.setdiff1d(covered_targets_, heldout_targets_)
    print("Total targets in literature: ", len(targets_in_literature_))
    print("Covered targets: ", len(covered_targets_))
    print("Heldout targets: ", len(heldout_targets_))
    print("Targets that can be used for training: ", len(valid_targets))
    print(
        f"Expected number of targets in training: ({perc_targets_in_training*100}% of {len(valid_targets)})",
        int(len(valid_targets) * perc_targets_in_training),
    )

    if perc_targets_in_training < 1.0:
        print("Sampling targets for training...")
        np.random.seed(random_seed)
        valid_targets = np.random.choice(
            valid_targets, size=int(len(valid_targets) * perc_targets_in_training), replace=False
        )
        print("Sampled targets for training: ", len(valid_targets))

        ref_db_ = ref_db_[ref_db_["target_gene"].str.lower().isin(valid_targets)]

    print("--------------------------------")
    print("Number of targets in training: ", len(ref_db_["target_gene"].unique()))

    Aweight = np.ones((adata.n_vars, adata.n_vars), dtype=np.float32)
    confidence_to_weight = {
        "S": 0.0,
        "C": 0.0,
        "W": 0.0,
        np.nan: 1.0,
    }

    for i, target_gene in enumerate(adata.var_names):
        for j, regulator_gene in enumerate(adata.var_names):
            target_gene_ = target_gene.lower()
            regulator_gene_ = regulator_gene.lower()
            key_ = f"{regulator_gene_}_{target_gene_}"
            if key_ in ref_db_.index:
                confidence_level = ref_db_.loc[key_, "confidenceLevel"]
                Aweight[i, j] = confidence_to_weight[confidence_level]
    return Aweight

In [ ]:
fitness_data_spacer = pd.read_csv(
    "/workspace/data/calvo2020_dcas9fitness/Supp_data2_log2FC.csv"
).rename(columns={"Unnamed: 0": "spacer"})
display(fitness_data_spacer.head())

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata.obs = adata.obs.merge(fitness_data_spacer, how="left", on="spacer")
adata.obs["spacer_has_fitness_data"] = adata.obs["spacer"].isin(fitness_data_spacer["spacer"])
adata.obs["spacer_is_control"] = (
    adata.obs["target"] == "nontargeting"
) | adata.obs.gene.str.startswith("Control")
adata.obs["spacer_is_valid"] = adata.obs["spacer_has_fitness_data"] | adata.obs["spacer_is_control"]

In [ ]:
adata = adata[~adata.obs["gene"].isna()].copy()
adata.obs.loc[lambda x: x["spacer_is_control"], "gene"] = "nontargeting"
adata.obs["consensus_target"] = adata.obs["gene"]

In [ ]:
gene_kos = adata.obs["consensus_target"].unique()
gene_kos = [g for g in gene_kos if g != "nontargeting"]
print(len(gene_kos))
gene_kos = [g for g in gene_kos if g in adata.var_names]
print(len(gene_kos))
np.random.seed(0)
heldout_perturbations = np.random.choice(gene_kos, size=500, replace=False)
training_perturbations = np.setdiff1d(gene_kos, heldout_perturbations)

adata = adata[adata.obs["consensus_target"].isin(gene_kos + ["nontargeting"])].copy()

In [ ]:
adata_dev = adata[~adata.obs["consensus_target"].isin(heldout_perturbations)].copy()
adata_test = adata[adata.obs["consensus_target"].isin(heldout_perturbations)].copy()

In [ ]:
from essential.configs.steady_state import get_config

base_config = get_config()
base_config.estimator.perturbation_col = "consensus_target"
base_config.estimator.control_key = "nontargeting"

In [ ]:
# mean_baseline
mean_results = []
errors = []
heldout_perturbations = adata_test.obs["consensus_target"].unique()
for g in heldout_perturbations:
    adata_test_u = adata_test[adata_test.obs["consensus_target"] == g].copy()
    X_gt_avg = adata_test_u.X.mean(0).A1
    X_pred = adata_dev.X.mean(0).A1
    X0 = adata_dev[adata_dev.obs["consensus_target"] == "nontargeting"].X.mean(0).A1
    dX_pred = X_pred - X0
    dX_gt = X_gt_avg - X0
    corrs = st.pearsonr(dX_pred, dX_gt)[0]
    L2_error = np.linalg.norm(X_pred - X_gt_avg)
    mean_results.append(corrs)
    errors.append(L2_error)
mean_results = pd.DataFrame({"correlation": mean_results, "error": errors}).assign(model="mean")
mean_results

# Sparsity prior effect

In [ ]:
results = []
# train_results = []

for lambda_prior in [
    1e-2,
    1e-1,
    1.0,
    10.0,
    100.0,
]:
    config = base_config.copy_and_resolve_references()
    config.estimator.model_kwargs.lambda_prior = lambda_prior
    estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
    estimator.fit(**config.training.to_dict())
    corrs, errors = test_model(estimator, adata_dev, adata_test)

    results.append(
        test_model(estimator, adata_dev, adata_test).assign(model=f"lambda={lambda_prior}")
    )
    # train_results.append(
    #     test_model(estimator, adata_dev, adata_dev).assign(model=f"lambda={lambda_prior}")
    # )
results = pd.concat(results)
# train_results = pd.concat(train_results)

In [ ]:
all_results = pd.concat([results, mean_results])
all_results["model"] = all_results["model"].str.replace("lambda=", "$\\lambda=$")

In [ ]:
(
    gg.ggplot(all_results)
    + gg.aes(x="model", y="error", fill="model")
    + gg.geom_boxplot()
    + gg.theme_minimal()
    + gg.theme(legend_position="none")
)

In [ ]:
(
    gg.ggplot(all_results)
    + gg.aes(x="model", y="correlation", fill="model")
    + gg.geom_boxplot()
    + gg.theme_minimal()
    + gg.theme(legend_position="none")
)

### # of epochs

In [ ]:
results = []

for n_epochs in [
    10,
    100,
    500,
    1000,
    5000,
    10000,
]:
    config = base_config.copy_and_resolve_references()
    config.training.n_epochs = n_epochs
    config.training.early_stopping_patience = 10000
    config.estimator.model_kwargs.lambda_prior = 100.0

    estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
    estimator.fit(**config.training.to_dict())
    corrs, errors = test_model(estimator, adata_dev, adata_test)

    results.append(
        test_model(estimator, adata_dev, adata_test).assign(model=f"n_epochs={n_epochs}")
    )
results = pd.concat(results)

In [ ]:
order = [f"n_epochs={n}" for n in [10, 100, 500, 1000, 5000, 10000]]
results["model"] = pd.Categorical(results["model"], categories=order, ordered=True)
(
    gg.ggplot(results)
    + gg.aes(x="model", y="correlation")
    + gg.geom_boxplot()
    + gg.geom_jitter(alpha=0.5)
    + gg.theme_minimal()
    + gg.theme(
        axis_text_x=gg.element_text(angle=20, hjust=1, vjust=0.5),
        axis_text_y=gg.element_text(hjust=1),
    )
    + gg.labs(x="")
)

### # of steps during inference

In [ ]:
results_n_steps = []
for n_steps in [2, 10, 25, 50, 100, 1000]:
    model_res = test_model(estimator, adata_dev, adata_test, n_steps=n_steps)
    results_n_steps.append(model_res.assign(n_steps=n_steps))
results_n_steps = pd.concat(results_n_steps)

In [ ]:
(
    gg.ggplot(results_n_steps)
    + gg.aes(x="factor(n_steps)", y="correlation")
    + gg.geom_boxplot()
    + gg.geom_jitter(alpha=0.5)
    + gg.theme_minimal()
    + gg.theme(
        axis_text_x=gg.element_text(angle=20, hjust=1, vjust=0.5),
        axis_text_y=gg.element_text(hjust=1),
    )
    + gg.labs(x="# of steps for inference")
)

# Lit knowledge

In [ ]:
Aweight = build_aweight(adata, heldout_targets=[], perc_targets_in_training=1.0, random_seed=0)
Amask = (Aweight == 0.0).astype(np.float32)

In [ ]:
LAMBDA_PRIOR = 100.0

results = pd.DataFrame()

config = base_config.copy_and_resolve_references()
config.estimator.model_kwargs.lambda_prior = LAMBDA_PRIOR
config.estimator.model_kwargs.Amask = None
config.estimator.model_kwargs.Aweight = None  # Safety
# config.training.n_epochs = 10
estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
estimator.fit(**config.training.to_dict())
# FIX: Hardcode lambda=0 in label or use the actual value
model_res = test_model(estimator, adata_dev, adata_test).assign(model="baseline")
results = pd.concat([results, model_res])


config = base_config.copy_and_resolve_references()
config.estimator.model_kwargs.lambda_prior = 0.0
config.estimator.model_kwargs.Amask = Amask
config.estimator.model_kwargs.Aweight = None  # Safety
# config.training.n_epochs = 10
estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
estimator.fit(**config.training.to_dict())
# FIX: Hardcode lambda=0 in label or use the actual value
model_res = test_model(estimator, adata_dev, adata_test).assign(
    model="lambda=0.0; masked interactions"
)
results = pd.concat([results, model_res])

# Block 2: Weighted Lasso (Soft Constraint)
config = base_config.copy_and_resolve_references()
config.estimator.model_kwargs.lambda_prior = LAMBDA_PRIOR
config.estimator.model_kwargs.Aweight = Aweight
config.estimator.model_kwargs.Amask = None  # Safety: Ensure no hard mask is applied
estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
estimator.fit(**config.training.to_dict())
model_res = test_model(estimator, adata_dev, adata_test).assign(
    model=f"lambda={LAMBDA_PRIOR}; weighted Lasso"
)
results = pd.concat([results, model_res])

# no lit knowledge
config = base_config.copy_and_resolve_references()
config.estimator.model_kwargs.lambda_prior = LAMBDA_PRIOR
estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
estimator.fit(**config.training.to_dict())
corrs, errors = test_model(estimator, adata_dev, adata_test)
model_res = test_model(estimator, adata_dev, adata_test).assign(model=f"lambda={LAMBDA_PRIOR}")

results = pd.concat([results, model_res])
all_results = pd.concat([results, mean_results])

In [ ]:
all_results = pd.concat([results, mean_results])
all_results = all_results.loc[all_results["model"] != "baseline"]
all_results["model"] = all_results["model"].str.replace("lambda=", "$\\lambda=$")

In [ ]:
(
    gg.ggplot(all_results)
    + gg.aes(x="model", y="error", fill="model")
    + gg.geom_boxplot()
    + gg.geom_jitter(alpha=0.5)
    + gg.theme_minimal()
    + gg.theme(
        axis_text_x=gg.element_text(angle=20),
        axis_text_y=gg.element_text(hjust=1),
        legend_position="none",
    )
    + gg.labs(x="")
)

In [ ]:
(
    gg.ggplot(all_results)
    + gg.aes(x="model", y="correlation", fill="model")
    + gg.geom_boxplot()
    + gg.geom_jitter(alpha=0.5)
    + gg.theme_minimal()
    + gg.theme(
        axis_text_x=gg.element_text(angle=20),
        axis_text_y=gg.element_text(hjust=1),
        legend_position="none",
    )
    + gg.labs(x="")
)

# RegulonDB vs LLM embeddings

In [ ]:
Aweight = build_aweight(adata, heldout_targets=[], perc_targets_in_training=1.0, random_seed=0)
Amask = (Aweight == 0.0).astype(np.float32)

In [ ]:
embedding_dir = "/workspace/data/e_coli_llm_embeddings"
llm_embeddings = np.load(os.path.join(embedding_dir, "llm_embeddings.npz"))
gene_order = pd.Series(np.arange(len(llm_embeddings["genes"])), index=llm_embeddings["genes"])

valid_genes = np.intersect1d(adata.var_names, llm_embeddings["genes"])
index_order = gene_order.loc[valid_genes].values
gene_embeddings = llm_embeddings["embeddings"][index_order]
d_embedding = gene_embeddings.shape[1]

adata_dev_llm = adata_dev[:, valid_genes].copy()
adata_test_llm = adata_test[:, valid_genes].copy()

In [ ]:
gene_embeddings_ = gene_embeddings.copy()
gene_embeddings_ = gene_embeddings_ - gene_embeddings_.mean(0)
gene_embeddings_ = gene_embeddings_ / gene_embeddings_.std(0)

In [ ]:
# from sklearn.decomposition import PCA

# pca = PCA(n_components=100)
# gene_embeddings_pca = pca.fit_transform(gene_embeddings_)

In [ ]:
# gene_embeddings_pca.shape

In [ ]:
# config = base_config.copy_and_resolve_references()
# config.estimator.model_class = "hardsigmoid2_embedding_steady_state"
# config.estimator.model_kwargs.lambda_prior = 0.0
# # config.estimator.model_kwargs.embeddings = gene_embeddings_
# # config.estimator.model_kwargs.embedding_dim = d_embedding
# config.estimator.model_kwargs.embeddings = gene_embeddings_pca
# config.estimator.model_kwargs.embedding_dim = 100
# config.training.n_epochs = 10000
# config.training.learning_rate = 1e-2

# estimator = SteadyStateEstimator(adata_dev_llm, **config.estimator.to_dict())
# estimator.fit(**config.training.to_dict())
# model_res = test_model(estimator, adata_dev_llm, adata_test_llm).assign(model=f"LLM embeddings PCA")
# results = pd.concat([results, model_res])

In [ ]:
# estimator.step_history_df["reco_loss"].plot()
# plt.yscale("log")
# plt.show()

In [ ]:
LAMBDA_PRIOR = 100.0

results = pd.DataFrame()

config = base_config.copy_and_resolve_references()
config.estimator.model_class = "hardsigmoid2_embedding_steady_state"
config.estimator.model_kwargs.lambda_prior = 0.0
config.estimator.model_kwargs.embeddings = gene_embeddings_
config.estimator.model_kwargs.embedding_dim = d_embedding
config.training.n_epochs = 10000
config.training.learning_rate = 1e-2

estimator = SteadyStateEstimator(adata_dev_llm, **config.estimator.to_dict())
estimator.fit(**config.training.to_dict())
model_res = test_model(estimator, adata_dev_llm, adata_test_llm).assign(model=f"LLM embeddings V2")
results = pd.concat([results, model_res])

# no lit knowledge
config = base_config.copy_and_resolve_references()
config.estimator.model_kwargs.lambda_prior = LAMBDA_PRIOR
estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
estimator.fit(**config.training.to_dict())
corrs, errors = test_model(estimator, adata_dev, adata_test)
model_res = test_model(estimator, adata_dev, adata_test).assign(model=f"lambda={LAMBDA_PRIOR}")
results = pd.concat([results, model_res])

# config = base_config.copy_and_resolve_references()
# config.estimator.model_kwargs.lambda_prior = 0.0
# config.estimator.model_kwargs.Amask = Amask
# config.estimator.model_kwargs.Aweight = None  # Safety
# # config.training.n_epochs = 10
# estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
# estimator.fit(**config.training.to_dict())
# # FIX: Hardcode lambda=0 in label or use the actual value
# model_res = test_model(estimator, adata_dev, adata_test).assign(
#     model="lambda=0.0; masked interactions"
# )
# results = pd.concat([results, model_res])

# # Block 2: Weighted Lasso (Soft Constraint)
# config = base_config.copy_and_resolve_references()
# config.estimator.model_kwargs.lambda_prior = LAMBDA_PRIOR
# config.estimator.model_kwargs.Aweight = Aweight
# config.estimator.model_kwargs.Amask = None  # Safety: Ensure no hard mask is applied
# estimator = SteadyStateEstimator(adata_dev, **config.estimator.to_dict())
# estimator.fit(**config.training.to_dict())
# model_res = test_model(estimator, adata_dev, adata_test).assign(
#     model=f"lambda={LAMBDA_PRIOR}; weighted Lasso"
# )
# results = pd.concat([results, model_res])

all_results = pd.concat([results, mean_results])

In [ ]:
all_results = pd.concat([results, mean_results])

In [ ]:
(
    gg.ggplot(all_results)
    + gg.aes(x="model", y="error")
    + gg.geom_boxplot()
    + gg.geom_jitter(alpha=0.5)
    + gg.theme_minimal()
    + gg.theme(
        axis_text_x=gg.element_text(angle=20),
        axis_text_y=gg.element_text(hjust=1),
    )
    + gg.labs(x="")
)